# Tutorial: Natural-Language-to-CEL Auto-Labeling Rules & `:test` Dry-Run Evaluation

This notebook demonstrates how to use `cxidk.wrapper.insights.AutoLabelingRules`, `cxidk.workflow.auto_labeling.CelRuleGenerator`, and `AutoLabelingPromoter` to:
1. **Compile Plain English to Schema-Verified `kJson` (`lowerCamelCase`) CEL**: Automatically generate `AutoLabelingRule` conditions with mandatory `has(...)` presence guards across `conversation.runtimeAnnotations`, `cesTurnAnnotation.messages`, `dialogflowInteraction.liveAgentHandoff`, and `conversation.transcript.transcriptSegments`.
2. **Static CEL Safety Validation & `snake_case` Auto-Repair**: Catch common CEL failures (`runtime_annotations` vs `runtimeAnnotations`, unquoted string literals, missing `has()` guards) before rule creation.
3. **Closed-Loop `:test` Dry-Run Evaluation**: Evaluate compiled rules against real or synthetic conversations via `POST /v1main/{parent}/autoLabelingRules:test` without mutating stored conversation data.
4. **Promote Rules Across Environments (`Dev → Prod`)**: Gate production promotion of `AutoLabelingRule` resources on a zero-error `:test` dry run.

In [ ]:
import json
from cxidk.workflow.auto_labeling import CelRuleGenerator

## 1. Compile Plain-English Queries into `kJson` (`lowerCamelCase`) CEL Rules

In Contact Center AI Insights, the C++ `AutoLabelingRuleEvaluator` converts `Conversation` protos into `google.protobuf.Struct` with `preserve_proto_field_names = false` (`CelExpressionFormat::kJson`). All fields must use `lowerCamelCase` and guard nested fields with `has(...)`.

In [ ]:
generator = CelRuleGenerator()

prompts = [
    (
        "billing_tool_failure",
        "Set to 'true' when the conversation was transferred to subagent 'billing_specialist' and any tool call returned an error.",
    ),
    (
        "customer_tier",
        "Extract the 'customer_tier' session parameter value with fallback 'standard'.",
    ),
    (
        "containment_outcome",
        "Label as escalated when a live agent handoff occurs in Dialogflow CX or CES containment fails.",
    ),
    (
        "churn_risk",
        "When the customer mentions 'cancel my account' in the transcript.",
    ),
]

for label_key, english_query in prompts:
    compiled = generator.from_english(english_prompt=english_query, label_key=label_key)
    print(f"=== Label Key: {label_key} (Pattern: {compiled.matched_pattern}) ===")
    print(json.dumps(compiled.to_api_dict(), indent=2))
    print()

## 2. Static CEL Safety Validator & Automatic `snake_case` Repair

If an engineer writes raw proto field names like `conversation.runtime_annotations` or forgets inner quotes on static values (`value: "true"` instead of `value: '"true"'`), `CelRuleGenerator.validate_cel_safety()` flags the exact error and `auto_fix_snake_case()` repairs the field names.

In [ ]:
broken_condition = "has(conversation.runtime_annotations) && conversation.turn_count > 10"
broken_value = "high_turns"

findings = CelRuleGenerator.validate_cel_safety(broken_condition, broken_value)
for f in findings:
    print(f"[{f.severity}] {f.rule_id}: {f.message}")
    print(f"  -> Remediation: {f.remediation}")

repaired_condition = CelRuleGenerator.auto_fix_snake_case(broken_condition)
print("\nRepaired condition:", repaired_condition)